# 05 — Final Data Load Preparation (Tableau Export)

This notebook generates the final CSV files optimised for Tableau Public.

**Strategy:**
- One loan-level sample (200k rows) for detail views
- Five pre-aggregated files for fast dashboard rendering

**Input:** `data/processed/lending_club_cleaned.csv`  
**Output:** 7 CSV files in `data/processed/`

In [ ]:
import pandas as pd
import os

df = pd.read_csv('../data/processed/lending_club_cleaned.csv', low_memory=False)
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")    

## 5.1 — Main Loan-Level File (200k Sample)

Tableau Public struggles with 2.9M rows. A stratified sample of 200k preserves the default rate distribution.

In [ ]:
tableau_cols = [
    'loan_default', 'loan_amnt', 'int_rate', 'grade',
    'term_n', 'emp_length_n', 'home_ownership', 'annual_inc',
    'verification_status', 'purpose', 'addr_state', 'dti',
    'fico_avg', 'revol_util', 'loan_to_income', 'credit_age_months',
    'issue_year', 'issue_month', 'delinq_flag', 'high_risk_flag',
    'mort_acc', 'pub_rec_bankruptcies', 'total_acc'
]

available_cols = [c for c in tableau_cols if c in df.columns]
missing_cols   = [c for c in tableau_cols if c not in df.columns]
if missing_cols:
    print("Skipping missing columns:", ", ".join(missing_cols))

tableau_df = df[available_cols].dropna(subset=['grade', 'addr_state', 'int_rate'])
sample_size = min(200_000, len(tableau_df))
tableau_sample = tableau_df.sample(n=sample_size, random_state=42)

tableau_sample.to_csv(f'{output_dir}/tableau_main.csv', index=False)
print(f"tableau_main.csv: {tableau_sample.shape}")    

## 5.2 — Aggregated: Default Rate by State (Map Chart)

In [ ]:
state_agg = df.groupby('addr_state').agg(
    total_loans    = ('loan_default', 'count'),
    total_defaults = ('loan_default', 'sum'),
    default_rate   = ('loan_default', 'mean'),
    avg_int_rate   = ('int_rate', 'mean'),
    avg_loan_amnt  = ('loan_amnt', 'mean'),
    avg_dti        = ('dti', 'mean')
).reset_index()
state_agg['default_rate'] = (state_agg['default_rate'] * 100).round(2)
state_agg['avg_int_rate'] = state_agg['avg_int_rate'].round(2)

state_agg.to_csv(f'{output_dir}/tableau_state.csv', index=False)
print(f"tableau_state.csv: {state_agg.shape[0]} states")    

## 5.3 — Aggregated: Default Rate by Year (Trend Chart)

In [ ]:
year_agg = df.groupby('issue_year').agg(
    total_loans    = ('loan_default', 'count'),
    total_defaults = ('loan_default', 'sum'),
    default_rate   = ('loan_default', 'mean'),
    avg_int_rate   = ('int_rate', 'mean'),
    avg_loan_amnt  = ('loan_amnt', 'mean')
).reset_index()
year_agg['default_rate'] = (year_agg['default_rate'] * 100).round(2)

year_agg.to_csv(f'{output_dir}/tableau_year.csv', index=False)
print(f"tableau_year.csv: {year_agg.shape[0]} years")
print(year_agg[['issue_year', 'total_loans', 'default_rate']])    

## 5.4 — Aggregated: Default Rate by Grade

In [ ]:
grade_agg = df.groupby('grade').agg(
    total_loans    = ('loan_default', 'count'),
    default_rate   = ('loan_default', 'mean'),
    avg_int_rate   = ('int_rate', 'mean'),
    avg_loan_amnt  = ('loan_amnt', 'mean')
).reset_index()
grade_agg['default_rate'] = (grade_agg['default_rate'] * 100).round(2)

grade_agg.to_csv(f'{output_dir}/tableau_grade.csv', index=False)
print(f"tableau_grade.csv:")
print(grade_agg)    

## 5.5 — Aggregated: Default Rate by Purpose

In [ ]:
purpose_agg = df.groupby('purpose').agg(
    total_loans  = ('loan_default', 'count'),
    default_rate = ('loan_default', 'mean'),
    avg_int_rate = ('int_rate', 'mean')
).reset_index()
purpose_agg['default_rate'] = (purpose_agg['default_rate'] * 100).round(2)

purpose_agg.to_csv(f'{output_dir}/tableau_purpose.csv', index=False)
print(f"tableau_purpose.csv: {purpose_agg.shape[0]} purposes")    

## 5.6 — Aggregated: Default Rate by Year + Grade

In [ ]:
year_grade = df.groupby(['issue_year', 'grade']).agg(
    total_loans  = ('loan_default', 'count'),
    default_rate = ('loan_default', 'mean'),
    avg_int_rate = ('int_rate', 'mean')
).reset_index()
year_grade['default_rate'] = (year_grade['default_rate'] * 100).round(2)

year_grade.to_csv(f'{output_dir}/tableau_year_grade.csv', index=False)
print(f"tableau_year_grade.csv: {year_grade.shape[0]} rows")    

## 5.7 — Aggregated: Default Rate by Year + Purpose

In [ ]:
year_purpose = df.groupby(['issue_year', 'purpose']).agg(
    total_loans  = ('loan_default', 'count'),
    default_rate = ('loan_default', 'mean')
).reset_index()
year_purpose['default_rate'] = (year_purpose['default_rate'] * 100).round(2)

year_purpose.to_csv(f'{output_dir}/tableau_year_purpose.csv', index=False)
print(f"tableau_year_purpose.csv: {year_purpose.shape[0]} rows")    

## Export Summary

In [ ]:
print("\n" + "=" * 60)
print("✅ ALL TABLEAU EXPORTS SAVED TO data/processed/")
print("=" * 60)
print("\nFILES TO IMPORT IN TABLEAU:")
print("  1. tableau_main.csv         → loan-level detail, filters, scatter plots")
print("  2. tableau_state.csv        → US map — default rate by state")
print("  3. tableau_year.csv         → trend line 2007–2020")
print("  4. tableau_grade.csv        → bar chart by grade")
print("  5. tableau_purpose.csv      → bar chart by purpose")
print("  6. tableau_year_grade.csv   → grade breakdown over time")
print("  7. tableau_year_purpose.csv → purpose breakdown over time")    